# 🏭 Industry 4.0 — Hybrid Offline Visual Quality Inspector
**Google Colab training and edge-export workbench — real dataset, no fabricated scores**

**What this builds (now using your dataset_manager.py registry):**
- **Known defects:** YOLO11n on selected `neu` (default), `gc10_hf`, properly annotated `pcb`/`gc10`, or factory YOLO dataset; mAP and stress tests.
- **Unknown anomalies:** lightweight **PatchCore-style nearest-neighbor feature memory** using ImageNet-pretrained MobileNetV3 intermediate features; trained and validated on **normal-only** MVTec AD *tile* images, or your own product images.
- **Decision safety:** image-quality gate, calibrated normal score threshold, uncertainty state, explainable severity policy (requires engineering approval), local SOP advice, event payloads and audit logs.
- **Export:** YOLO ONNX, MobileNet feature extractor ONNX, normal-patch bank and JSON configuration for Raspberry Pi.
- **Actual on-device Llama requirement:** `edge_inspector.py` can asynchronously call `llama_main` with a `.pte` Llama 3.2 1B program on the Pi's Arm CPU using **ExecuTorch**. This notebook *does not* pretend to export Llama in Colab.

> **CRITICAL:** NEU-DET steel images and MVTec *tile* images are **different products**. Train/evaluate each branch in its own domain. Do **not** fuse their scores on one product. A production fusion model requires *both* a defect detector and a normal-data anomaly model trained on **the SAME product/camera/lighting family**.

Run **Runtime → Change runtime type → T4 GPU** (recommended), then **Runtime → Run all**. The NEU detector uses live Kaggle download; the anomaly demo uses a public Hugging Face mirror. Network access is needed during Colab training/download only; exported inference is offline. Check licensing (MVTec non-commercial; Ultralytics and Llama licensing) before deployment.
**NEW:** The notebook embeds and uses the exact `dataset_manager.py` you uploaded; data comes from its dataset IDs. Select the detector source and anomaly/classification tasks in the configuration cell. GC10-HF, NEU VOC, generic VOC and labeled YOLO manifests have separate adapters; mask-only/class-only datasets are NOT passed off as YOLO boxes.


## 0. Architecture and evaluation plan

```text
Camera → part-ID/ROI → image-quality gate → [detector | normal-only anomaly]
        → threshold/calibration → rules/REVIEW → local SOP retrieval
        → Llama 3.2 1B via ExecuTorch ON PI CPU → offline SQLite audit
```
**Two data tracks:** NEU-DET is a detector benchmark; MVTec AD tile is a normal/anomaly benchmark. No misleading joint accuracy score. For a real deployment, acquire target-product normal images, defect labels, lighting/batch identifiers, and an approved acceptance policy.

**Metrics:** detection mAP50/mAP50–95, class recall, stress degradation; anomaly image AUROC and false alarms on held-out good; field tests false accepts, false rejects, FP per 1,000 good units, 95th percentile latency, per-shift drift. **Do not claim factory-ready accuracy based on public data alone.**

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "kagglehub>=0.3" "kaggle>=1.7" "datasets>=3.1,<6" "huggingface_hub>=0.26" "scikit-learn>=1.4" "onnx>=1.16" "onnxruntime>=1.20" "opencv-python-headless>=4.8" "PyYAML>=6" "joblib>=1.3"

In [ ]:
import os, gc, json, random, shutil, time, math, warnings, hashlib, zipfile, io
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, cv2, yaml, torch, matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from IPython.display import display
from ultralytics import YOLO

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
ROOT=Path('/content/industry4_qc'); ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)  # dataset_manager.py stores data/raw and data/processed relative to CWD
DATA_MODE='neu'              # choices: neu, gc10_hf, pcb, gc10, pcb_small, custom
CUSTOM_YAML='/content/my_factory_dataset/data.yaml'
# anomaly demo: 'mvtec_stream_demo' (default quick benchmark), 'mvtec_manual', 'custom'
ANOMALY_SOURCE='mvtec_stream_demo'
MVTEC_CATEGORY='tile'
MVTEC_MANUAL_ROOT=ROOT/'data'/'raw'/'mvtec'  # download from dataset_manager.py manual
# Optional stand-alone classification route for class-folder datasets.
CLASSIFICATION_SOURCE='none'  # 'magnetic_tile' or 'none'; don't merge with YOLO box classes
CLASSIFICATION_EPOCHS=12
EPOCHS=18                   # demonstration; tune 60-120+ for a serious experiment
IMGSZ=640                   # small thin cracks need high effective pixels / optional tiling
BATCH=16 if torch.cuda.is_available() else 4
DEVICE=0 if torch.cuda.is_available() else 'cpu'
MODEL_NAME='yolo11n.pt'     # well-supported detection/Hailo baseline
RUN_ANOMALY_DEMO=True
ANOMALY_PRODUCT='mvtec_tile' # NEVER re-use as steel_neu anomaly bank
MAX_MVTEC_TRAIN=90
MAX_MVTEC_VAL=30
MAX_MVTEC_TEST_PER_CLASS=45
NORMAL_DIR=ROOT/'my_product'/'normal'  # optional target-domain good samples, see later
print('Torch',torch.__version__, '| device',DEVICE,'| root',ROOT)

## Dataset manager from your upload — integrated into this notebook

The next cell writes **your supplied `dataset_manager.py` source** into the Colab project folder. Its Kaggle/Hugging Face IDs, manual dataset registry, folder summary, MVTec PyTorch loader and classification organizer are preserved. The *following* cells route selected datasets to appropriate ML tasks. Choose **only one defect-localization dataset at a time**; class taxonomies are not interoperable.

**Registry routing:** `neu`, `pcb`, `pcb_small`, `gc10`, `gc10_hf` → localization **if boxes/YOLO labels exist**; `magnetic_tile` → optional image classification; `mvtec_hf`/manual MVTec → anomaly; `severstal_hf` → segmentation only; `adnet` → optional, intentionally not auto-downloaded. A registry item being listed does **not** mean the adapter can automatically interpret every host's folder format.

**Downloads require Colab internet**; production Raspberry Pi inference does not.

In [ ]:
%%writefile /content/industry4_qc/dataset_manager.py
#!/usr/bin/env python3
"""
dataset_manager.py

Utility script for manufacturing defect datasets.

Features:
- Download supported datasets from Kaggle and Hugging Face.
- Print manual dataset download links.
- Organize classification datasets into train/val/test.
- Summarize image folders.
- Load MVTec-style anomaly detection datasets with PyTorch.

Usage examples:
    python dataset_manager.py list
    python dataset_manager.py manual
    python dataset_manager.py download neu pcb magnetic_tile gc10
    python dataset_manager.py organize-classification --source data/raw/neu --dest data/processed/classification/neu
    python dataset_manager.py summarize --source data/raw/neu
    python dataset_manager.py demo-anomaly --root data/raw/mvtec --category hazelnut
"""

from __future__ import annotations

import argparse
import random
import shutil
import subprocess
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ---------------------------------------------------------------------
# Basic paths
# ---------------------------------------------------------------------

RAW = Path("data/raw")
PROCESSED = Path("data/processed")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# ---------------------------------------------------------------------
# Dataset registry
# ---------------------------------------------------------------------

DATASETS: Dict[str, Dict[str, str]] = {
    "neu": {
        "source": "kaggle",
        "id": "kaustubhdikshit/neu-surface-defect-database",
        "desc": "NEU steel surface classification dataset, 1800 images",
    },
    "pcb": {
        "source": "kaggle",
        "id": "norbertelter/pcb-defect-dataset",
        "desc": "PCB defect dataset, augmented, around 10668 images",
    },
    "pcb_small": {
        "source": "kaggle",
        "id": "akhatova/pcb-defects",
        "desc": "Synthetic PCB defect dataset, 1386 images",
    },
    "magnetic_tile": {
        "source": "kaggle",
        "id": "alex000kim/magnetic-tile-surface-defects",
        "desc": "Magnetic tile surface defect dataset",
    },
    "gc10": {
        "source": "kaggle",
        "id": "alex000kim/gc10det",
        "desc": "GC10-DET steel surface defect detection dataset",
    },
    "gc10_hf": {
        "source": "hf",
        "id": "dronefreak/GC10-DET",
        "desc": "GC10-DET dataset hosted on Hugging Face",
    },
    "severstal_hf": {
        "source": "hf",
        "id": "Voxel51/severstal_steel_defects",
        "desc": "Severstal steel defect segmentation dataset hosted on Hugging Face",
    },
    "mvtec_hf": {
        "source": "hf",
        "id": "Voxel51/mvtec-ad",
        "desc": "MVTec AD hosted on Hugging Face. Official site may require manual download.",
    },
    "adnet": {
        "source": "hf",
        "id": "linglingling009/ADNet",
        "desc": "ADNet large anomaly detection dataset, 196k images. Very large.",
    },
}

HUGE_DATASETS = {"adnet"}

MANUAL_DATASETS: Dict[str, str] = {
    "MVTec AD official": "https://www.mvtec.com/company/research/datasets/mvtec-ad",
    "MVTec AD 2": "https://www.mvtec.com/company/research/datasets/mvtec-ad-2",
    "MVTec LOCO AD": "https://www.mvtec.com/company/research/datasets/mvtec-loco-ad",
    "MVTec 3D-AD": "https://www.mvtec.com/company/research/datasets/mvtec-3d-ad",
    "VisA AWS Open Data": "https://registry.opendata.aws/visa/",
    "KolektorSDD2": "https://www.vicos.si/resources/kolektorsdd2/",
    "Real-IAD": "https://realiad4ad.github.io/Real-IAD/",
    "Kaputt": "https://www.kaputt-dataset.com/",
    "NEU official": "https://faculty.neu.edu.cn/songkc/en/zdylm/263265/",
}

# ---------------------------------------------------------------------
# Optional PyTorch imports
# ---------------------------------------------------------------------

try:
    import torch
    from torch.utils.data import Dataset, DataLoader
    from torchvision import transforms
    from PIL import Image

    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def ensure_dirs() -> None:
    RAW.mkdir(parents=True, exist_ok=True)
    PROCESSED.mkdir(parents=True, exist_ok=True)


def is_image(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTENSIONS


def run_command(cmd: List[str]) -> None:
    print("$", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, check=True)


# ---------------------------------------------------------------------
# Dataset listing
# ---------------------------------------------------------------------

def list_datasets() -> None:
    print("=" * 80)
    print("Automatic datasets available through this script")
    print("=" * 80)

    for name, cfg in DATASETS.items():
        huge = " [HUGE]" if name in HUGE_DATASETS else ""
        print(f"\n{name}{huge}")
        print(f"  source: {cfg['source']}")
        print(f"  id:     {cfg['id']}")
        print(f"  desc:   {cfg['desc']}")

    print("\nUse:")
    print("  python dataset_manager.py download <dataset_name> ...")
    print("\nExample:")
    print("  python dataset_manager.py download neu pcb magnetic_tile gc10")


def print_manual_datasets() -> None:
    print("=" * 80)
    print("Manual datasets / official pages")
    print("=" * 80)

    for name, url in MANUAL_DATASETS.items():
        print(f"\n{name}")
        print(f"  {url}")

    print("\nSome datasets require accepting licenses or filling forms.")
    print("Download them manually and place them under data/raw/.")


# ---------------------------------------------------------------------
# Download functions
# ---------------------------------------------------------------------

def download_kaggle_dataset(name: str, dataset_id: str) -> None:
    dest = RAW / name
    dest.mkdir(parents=True, exist_ok=True)

    try:
        run_command(
            [
                "kaggle",
                "datasets",
                "download",
                "-d",
                dataset_id,
                "-p",
                str(dest),
                "--unzip",
            ]
        )
        print(f"[OK] Downloaded Kaggle dataset '{name}' to {dest}")
    except FileNotFoundError:
        sys.exit(
            "Kaggle CLI not found.\n"
            "Install it with: pip install kaggle\n"
            "Then place your kaggle.json in ~/.kaggle/kaggle.json"
        )
    except subprocess.CalledProcessError as exc:
        print(f"[ERROR] Failed to download Kaggle dataset '{name}': {exc}")


def download_hf_dataset(name: str, repo_id: str) -> None:
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        sys.exit("huggingface_hub not found. Install it with: pip install huggingface_hub")

    dest = RAW / name
    dest.mkdir(parents=True, exist_ok=True)

    try:
        print(f"Downloading Hugging Face dataset: {repo_id}")
        print(f"Destination: {dest}")

        snapshot_download(
            repo_id=repo_id,
            repo_type="dataset",
            local_dir=str(dest),
        )

        print(f"[OK] Downloaded Hugging Face dataset '{name}' to {dest}")

    except Exception as exc:
        print(f"[ERROR] Failed to download Hugging Face dataset '{name}': {exc}")
        print("If the dataset is gated, run: huggingface-cli login")


def download_datasets(names: List[str], assume_yes: bool = False) -> None:
    ensure_dirs()

    for name in names:
        cfg = DATASETS.get(name)

        if cfg is None:
            print(f"[WARN] Unknown dataset: {name}")
            continue

        if name in HUGE_DATASETS and not assume_yes:
            print(f"[SKIP] '{name}' is very large. Use --yes to download it.")
            continue

        print("\n" + "=" * 80)
        print(f"Downloading: {name}")
        print("=" * 80)

        if cfg["source"] == "kaggle":
            download_kaggle_dataset(name, cfg["id"])
        elif cfg["source"] == "hf":
            download_hf_dataset(name, cfg["id"])
        else:
            print(f"[WARN] Unsupported source for dataset: {name}")


# ---------------------------------------------------------------------
# Classification dataset organization
# ---------------------------------------------------------------------

def organize_classification_dataset(
    source: Path,
    dest: Path,
    seed: int = 42,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
) -> None:
    """
    Organize a classification dataset into:

        dest/
            train/<class>/...
            val/<class>/...
            test/<class>/...

    Expected input structure:

        source/
            class_1/
                image1.png
                image2.png
            class_2/
                image3.png
    """

    source = Path(source)
    dest = Path(dest)

    if not source.exists():
        sys.exit(f"Source folder does not exist: {source}")

    class_dirs = [
        d
        for d in source.iterdir()
        if d.is_dir() and any(is_image(f) for f in d.iterdir() if f.is_file())
    ]

    if not class_dirs:
        sys.exit(
            "No class folders found.\n"
            "Expected structure: source/<class_name>/images"
        )

    random.seed(seed)

    print(f"Organizing classification dataset")
    print(f"Source: {source}")
    print(f"Destination: {dest}")

    for class_dir in class_dirs:
        files = [
            f
            for f in class_dir.iterdir()
            if f.is_file() and is_image(f)
        ]

        random.shuffle(files)

        n_train = int(len(files) * train_ratio)
        n_val = int(len(files) * val_ratio)

        train_files = files[:n_train]
        val_files = files[n_train:n_train + n_val]
        test_files = files[n_train + n_val:]

        splits = {
            "train": train_files,
            "val": val_files,
            "test": test_files,
        }

        for split_name, split_files in splits.items():
            out_dir = dest / split_name / class_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)

            for i, file_path in enumerate(split_files):
                out_name = f"{i:06d}_{file_path.name}"
                shutil.copy2(file_path, out_dir / out_name)

        print(
            f"[OK] {class_dir.name}: "
            f"train={len(train_files)}, "
            f"val={len(val_files)}, "
            f"test={len(test_files)}"
        )


# ---------------------------------------------------------------------
# Dataset summarizer
# ---------------------------------------------------------------------

def summarize_folder(source: Path) -> None:
    source = Path(source)

    if not source.exists():
        sys.exit(f"Folder does not exist: {source}")

    counts: Dict[str, int] = defaultdict(int)

    for path in source.rglob("*"):
        if path.is_file() and is_image(path):
            rel_parts = path.relative_to(source).parts
            key = "/".join(rel_parts[:-1]) if len(rel_parts) > 1 else "root"
            counts[key] += 1

    if not counts:
        print(f"No images found in {source}")
        return

    print("=" * 80)
    print(f"Image summary for: {source}")
    print("=" * 80)

    total = 0
    for key in sorted(counts.keys()):
        print(f"{counts[key]:8d}  {key}")
        total += counts[key]

    print("-" * 80)
    print(f"{total:8d}  TOTAL")


# ---------------------------------------------------------------------
# PyTorch anomaly dataset loader, MVTec-style
# ---------------------------------------------------------------------

if TORCH_AVAILABLE:

    def default_transform(image_size: int = 256):
        return transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225),
                ),
            ]
        )

    def default_mask_transform(image_size: int = 256):
        return transforms.Compose(
            [
                transforms.Resize(
                    (image_size, image_size),
                    interpolation=transforms.InterpolationMode.NEAREST,
                ),
                transforms.ToTensor(),
            ]
        )

    class MVTecADDataset(Dataset):
        """
        Dataset loader for MVTec-style anomaly detection datasets.

        Expected folder structure:

            root/
                category/
                    train/
                        good/
                            image.png
                    test/
                        good/
                            image.png
                        defect_name/
                            image.png
                    ground_truth/
                        defect_name/
                            image_mask.png
        """

        def __init__(
            self,
            root: str,
            category: str,
            split: str = "train",
            image_size: int = 256,
            transform=None,
            mask_transform=None,
        ):
            self.root = Path(root) / category
            self.category = category
            self.split = split
            self.image_size = image_size

            if not self.root.exists():
                raise FileNotFoundError(
                    f"Dataset category folder not found: {self.root}\n"
                    "Expected structure: root/<category>/train/good and root/<category>/test/..."
                )

            self.transform = transform or default_transform(image_size)
            self.mask_transform = mask_transform or default_mask_transform(image_size)

            self.samples: List[Tuple[Path, Optional[Path], int, str]] = []
            self._build_samples()

        @staticmethod
        def _is_image(path: Path) -> bool:
            return path.suffix.lower() in IMAGE_EXTENSIONS

        def _find_mask(self, defect_type: str, image_path: Path) -> Optional[Path]:
            gt_dir = self.root / "ground_truth" / defect_type

            if not gt_dir.exists():
                return None

            candidates = [
                gt_dir / f"{image_path.stem}_mask.png",
                gt_dir / image_path.name,
            ]

            for ext in IMAGE_EXTENSIONS:
                candidates.append(gt_dir / f"{image_path.stem}_mask{ext}")
                candidates.append(gt_dir / f"{image_path.stem}{ext}")

            for candidate in candidates:
                if candidate.exists():
                    return candidate

            return None

        def _build_samples(self) -> None:
            if self.split not in {"train", "test"}:
                raise ValueError("split must be 'train' or 'test'")

            if self.split == "train":
                good_dir = self.root / "train" / "good"

                if not good_dir.exists():
                    raise FileNotFoundError(
                        f"Training good folder not found: {good_dir}"
                    )

                for path in sorted(good_dir.iterdir()):
                    if self._is_image(path):
                        self.samples.append((path, None, 0, "good"))

            else:
                test_dir = self.root / "test"

                if not test_dir.exists():
                    raise FileNotFoundError(f"Test folder not found: {test_dir}")

                for defect_dir in sorted(test_dir.iterdir()):
                    if not defect_dir.is_dir():
                        continue

                    defect_type = defect_dir.name
                    label = 0 if defect_type == "good" else 1

                    for path in sorted(defect_dir.iterdir()):
                        if not self._is_image(path):
                            continue

                        mask_path = None
                        if label == 1:
                            mask_path = self._find_mask(defect_type, path)

                        self.samples.append((path, mask_path, label, defect_type))

        def __len__(self) -> int:
            return len(self.samples)

        def __getitem__(self, idx: int):
            image_path, mask_path, label, defect_type = self.samples[idx]

            image = Image.open(image_path).convert("RGB")

            if self.transform is not None:
                image = self.transform(image)

            if mask_path is not None:
                mask = Image.open(mask_path).convert("L")

                if self.mask_transform is not None:
                    mask = self.mask_transform(mask)

                mask = (mask > 0.0).float()
            else:
                mask = torch.zeros(image.shape[-2:], dtype=torch.float32)

            return {
                "image": image,
                "mask": mask,
                "label": label,
                "defect_type": defect_type,
                "image_path": str(image_path),
            }

    def make_anomaly_dataloaders(
        root: str,
        category: str,
        batch_size: int = 8,
        image_size: int = 256,
        num_workers: int = 0,
    ):
        train_dataset = MVTecADDataset(
            root=root,
            category=category,
            split="train",
            image_size=image_size,
        )

        test_dataset = MVTecADDataset(
            root=root,
            category=category,
            split="test",
            image_size=image_size,
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
        )

        return train_loader, test_loader


def demo_anomaly_dataset(
    root: str,
    category: str,
    image_size: int,
    batch_size: int,
) -> None:
    if not TORCH_AVAILABLE:
        sys.exit("PyTorch is not installed. Install torch and torchvision to use this feature.")

    train_loader, test_loader = make_anomaly_dataloaders(
        root=root,
        category=category,
        batch_size=batch_size,
        image_size=image_size,
        num_workers=0,
    )

    print("=" * 80)
    print("Anomaly dataset demo")
    print("=" * 80)
    print(f"Root:      {root}")
    print(f"Category:  {category}")
    print(f"Train batches: {len(train_loader)}")
    print(f"Test batches:  {len(test_loader)}")

    try:
        batch = next(iter(train_loader))
        print("\nTrain batch:")
        print("  image shape:", batch["image"].shape)
        print("  mask shape: ", batch["mask"].shape)
        print("  labels:     ", batch["label"])
        print("  defect type:", batch["defect_type"])
    except StopIteration:
        print("\nNo training samples found.")

    try:
        batch = next(iter(test_loader))
        print("\nTest batch:")
        print("  image shape:", batch["image"].shape)
        print("  mask shape: ", batch["mask"].shape)
        print("  labels:     ", batch["label"])
        print("  defect type:", batch["defect_type"])
    except StopIteration:
        print("\nNo test samples found.")


# ---------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------

def main() -> None:
    parser = argparse.ArgumentParser(
        description="Dataset manager for manufacturing defect datasets."
    )

    subparsers = parser.add_subparsers(dest="cmd")

    # list
    subparsers.add_parser("list", help="List automatic datasets")

    # manual
    subparsers.add_parser("manual", help="Show manual dataset links")

    # download
    download_parser = subparsers.add_parser("download", help="Download datasets")
    download_parser.add_argument(
        "names",
        nargs="+",
        help="Dataset names to download",
    )
    download_parser.add_argument(
        "--yes",
        action="store_true",
        help="Allow downloading very large datasets",
    )

    # organize-classification
    organize_parser = subparsers.add_parser(
        "organize-classification",
        help="Organize classification dataset into train/val/test",
    )
    organize_parser.add_argument("--source", required=True)
    organize_parser.add_argument("--dest", required=True)
    organize_parser.add_argument("--seed", type=int, default=42)

    # summarize
    summarize_parser = subparsers.add_parser(
        "summarize",
        help="Count images in folders",
    )
    summarize_parser.add_argument("--source", required=True)

    # demo-anomaly
    demo_parser = subparsers.add_parser(
        "demo-anomaly",
        help="Load MVTec-style anomaly dataset and print batch shapes",
    )
    demo_parser.add_argument("--root", default=str(RAW / "mvtec_hf"))
    demo_parser.add_argument("--category", default="hazelnut")
    demo_parser.add_argument("--image-size", type=int, default=256)
    demo_parser.add_argument("--batch-size", type=int, default=4)

    args = parser.parse_args()

    if args.cmd == "list":
        list_datasets()

    elif args.cmd == "manual":
        print_manual_datasets()

    elif args.cmd == "download":
        download_datasets(args.names, assume_yes=args.yes)

    elif args.cmd == "organize-classification":
        organize_classification_dataset(
            source=Path(args.source),
            dest=Path(args.dest),
            seed=args.seed,
        )

    elif args.cmd == "summarize":
        summarize_folder(Path(args.source))

    elif args.cmd == "demo-anomaly":
        demo_anomaly_dataset(
            root=args.root,
            category=args.category,
            image_size=args.image_size,
            batch_size=args.batch_size,
        )

    else:
        parser.print_help()


if __name__ == "__main__":
    main()

## 1. Choose and download a dataset from your `dataset_manager.py` registry

**Default:** NEU for 6-class steel defect bounding-box detection. `gc10_hf` has a verified YOLO-compatible `data/images/{train,valid,test}` and corresponding label tree; `pcb`/`gc10`/`pcb_small` can be used only if their downloaded files contain usable YOLO split metadata or matching VOC XML annotations. Generic conversion checks the content and **raises clearly** for unsupported layouts rather than fabricating labels. 

- For Kaggle data: upload `kaggle.json` to Colab or set Kaggle environment credentials. `kagglehub` public fallback is also attempted.
- For HF: `gc10_hf` downloads the registered repository; it is a relatively large download.
- If a source is already present under `data/raw/<dataset>`, the download is skipped.
- Do **not** train on `severstal_hf` segmentation masks as if they were YOLO boxes.


In [ ]:
import dataset_manager as dm
from collections import defaultdict
from sklearn.model_selection import train_test_split

# Display user's dataset registry without downloading everything.
print('User-supplied dataset registry:')
dm.list_datasets()
print('Manual dataset URLs are available via dm.print_manual_datasets()')

DETECTION_SOURCES={'neu','gc10_hf','gc10','pcb','pcb_small','custom'}
if DATA_MODE not in DETECTION_SOURCES:
    raise ValueError(f'{DATA_MODE}: not a valid detection source. Segmentation/anomaly/classification require separate tasks.')

def download_from_registry(name):
    dest=ROOT/'data'/'raw'/name
    dest.mkdir(parents=True,exist_ok=True)
    # Empty dirs should not be mistaken for successful downloads.
    if any(p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.bmp','.png','.xml','.txt','.yaml','.json','.parquet')
           for p in dest.rglob('*')):
        print('Using existing data:',dest)
        return dest
    cfg=dm.DATASETS[name]
    if cfg['source']=='kaggle':
        try:
            dm.download_kaggle_dataset(name,cfg['id'])
        except (SystemExit,Exception) as e:
            # Public fallback, not a replacement for user's Kaggle registry.
            print('Kaggle CLI failed; trying public kagglehub downloader:',str(e)[:180])
            import kagglehub
            downloaded=Path(kagglehub.dataset_download(cfg['id']))
            for item in downloaded.iterdir():
                target=dest/item.name
                if item.is_dir(): shutil.copytree(item,target,dirs_exist_ok=True)
                elif not target.exists(): shutil.copy2(item,target)
        # Original manager handles Kaggle failures internally. Check whether any real
        # files arrived before falling back, even if it printed an error and returned.
        if not any(p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.bmp','.png','.xml','.yaml','.txt')
                   for p in dest.rglob('*')):
            import kagglehub
            print('No Kaggle files after CLI; attempting kagglehub public download.')
            downloaded=Path(kagglehub.dataset_download(cfg['id']))
            for item in downloaded.iterdir():
                target=dest/item.name
                if item.is_dir(): shutil.copytree(item,target,dirs_exist_ok=True)
                elif not target.exists(): shutil.copy2(item,target)
    else:
        # For GC10-DET, only the data tree is needed. The registered repo also
        # contains large supplementary content; keep it for optional use later.
        if name=='gc10_hf':
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id=cfg['id'],repo_type='dataset',local_dir=str(dest),
                              allow_patterns=['data/images/**','data/labels/**','data/data.yaml'])
        else:
            dm.download_hf_dataset(name,cfg['id'])
    count=sum(1 for p in dest.rglob('*') if p.is_file())
    if not any(p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.bmp','.png') for p in dest.rglob('*')):
        raise RuntimeError(f'{name}: No images found after download at {dest}. Check Kaggle credentials or dataset layout.')
    print('Downloaded/available',name,'files',count)
    return dest

CLASSES=['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']
ALIASES={'pitted-surface':'pitted_surface','pitted surface':'pitted_surface',
         'rolled_in_scale':'rolled-in_scale','rolled in scale':'rolled-in_scale',
         'rolled-in scale':'rolled-in_scale','scratch':'scratches'}
def canonical_label(x):
    x=x.lower().strip()
    return ALIASES.get(x,x)

if DATA_MODE!='custom':
    raw=download_from_registry(DATA_MODE)
    dm.summarize_folder(raw)
else:
    raw=None

if DATA_MODE=='neu':
    all_xml=sorted(raw.rglob('*.xml'))
    all_images=[p for p in raw.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.bmp','.png')]
    # Store all same-stem candidates so a different split cannot be silently overwritten.
    image_candidates=defaultdict(list)
    for p in all_images: image_candidates[p.stem.lower()].append(p)
    image_map={}
    ambiguous=[]
    for stem,paths in image_candidates.items():
        if len(paths)==1: image_map[stem]=paths[0]
        else: ambiguous.append((stem,[str(x) for x in paths[:5]]))
    print('NEU images:',len(all_images),'XML:',len(all_xml),'ambiguous filename stems:',len(ambiguous))
    if ambiguous:
        raise ValueError(f'NEU duplicated image stems; inspect pairs instead of leaking/misaligning samples: {ambiguous[:3]}')
    if len(all_xml)<50: raise ValueError('This NEU mirror lacks VOC XML bounding boxes. Use classification task or choose a detection mirror.')


In [ ]:
import xml.etree.ElementTree as ET

def parse_voc(xml):
    node=ET.parse(xml).getroot()
    size=node.find('size')
    W=int(size.findtext('width')) if size is not None else 0
    H=int(size.findtext('height')) if size is not None else 0
    target_image=image_map.get(xml.stem.lower())
    if target_image is None: return None
    img=cv2.imread(str(target_image))
    if img is None: return None
    H,W=img.shape[:2] # ground truth image dimensions take priority
    boxes=[]
    for obj in node.findall('object'):
        label=canonical_label(obj.findtext('name',''))
        if label not in CLASSES: continue
        b=obj.find('bndbox')
        if b is None: continue
        try:
            x1=max(0.,min(float(b.findtext('xmin')),W-1.))
            y1=max(0.,min(float(b.findtext('ymin')),H-1.))
            x2=max(0.,min(float(b.findtext('xmax')),float(W)))
            y2=max(0.,min(float(b.findtext('ymax')),float(H)))
        except (TypeError,ValueError): continue
        if x2<=x1 or y2<=y1: continue
        boxes.append((CLASSES.index(label),x1,y1,x2,y2))
    return {'image':target_image,'boxes':boxes,'width':W,'height':H,
            'stratum':CLASSES[boxes[0][0]] if boxes else 'unlabeled'}

if DATA_MODE=='neu':
    records=[r for x in all_xml if (r:=parse_voc(x)) is not None]
    print('Matched images',len(records), 'Boxes',sum(len(r['boxes']) for r in records))
    print('By primary class',Counter(r['stratum'] for r in records))
    assert len(records)>=500 and sum(len(r['boxes']) for r in records)>1000
    sample=random.sample(records,min(6,len(records)))
    fig,axs=plt.subplots(2,3,figsize=(13,8))
    for ax,r in zip(axs.ravel(),sample):
        img=cv2.imread(str(r['image']))
        for cls,x1,y1,x2,y2 in r['boxes']:
            cv2.rectangle(img,(int(x1),int(y1)),(int(x2),int(y2)),(0,255,0),1)
            cv2.putText(img,CLASSES[cls],(int(x1),max(12,int(y1))),cv2.FONT_HERSHEY_SIMPLEX,.35,(0,255,0),1)
        ax.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB)); ax.axis('off')
    plt.tight_layout(); plt.show()

## 2. Leak-aware train/validation/test split and VOC→YOLO conversion

NEU lacks manufacturing batch/date IDs. This stratified split is **only a public-dataset benchmark**. For your factory dataset **split entire batches, days, suppliers, components and camera sessions**, not individual frames from the same part. Do not tune on the final test split.

In [ ]:
if DATA_MODE=='neu':
    train_rows,holdout=train_test_split(records,test_size=.30,random_state=SEED,stratify=[r['stratum'] for r in records])
    val_rows,test_rows=train_test_split(holdout,test_size=.50,random_state=SEED,stratify=[r['stratum'] for r in holdout])
    DATA_ROOT=ROOT/'neu_yolo'
    for split in ('train','val','test'):
        (DATA_ROOT/'images'/split).mkdir(parents=True,exist_ok=True)
        (DATA_ROOT/'labels'/split).mkdir(parents=True,exist_ok=True)
    for split,rows in [('train',train_rows),('val',val_rows),('test',test_rows)]:
        for idx,r in enumerate(rows):
            name=f'{r["image"].stem}_{idx:04d}'
            shutil.copy2(r['image'], DATA_ROOT/'images'/split/f'{name}{r["image"].suffix.lower()}')
            labels=[]
            for cls,x1,y1,x2,y2 in r['boxes']:
                cx=(x1+x2)/(2*r['width']); cy=(y1+y2)/(2*r['height'])
                w=(x2-x1)/r['width']; h=(y2-y1)/r['height']
                labels.append(f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
            (DATA_ROOT/'labels'/split/f'{name}.txt').write_text('\n'.join(labels))
    DATA_YAML=ROOT/'neu_yolo.yaml'
    DATA_YAML.write_text(yaml.safe_dump({'path':str(DATA_ROOT),'train':'images/train',
                                        'val':'images/val','test':'images/test','names':CLASSES},sort_keys=False))
    print('Train',len(train_rows),'Val',len(val_rows),'Test',len(test_rows),'\n',DATA_YAML.read_text())
    assert set(r['image'] for r in train_rows).isdisjoint(r['image'] for r in test_rows)
else:
    import xml.etree.ElementTree as ET
    def find_existing_yolo(root):
        # Do not assume filenames imply label type. Require a names map and split images/labels.
        for config in sorted(list(root.rglob('data.yaml'))+list(root.rglob('dataset.yaml'))):
            try: cfg=yaml.safe_load(config.read_text()) or {}
            except Exception: continue
            if not {'train','val','names'}.issubset(cfg): continue
            base=Path(cfg.get('path',config.parent))
            if not base.is_absolute(): base=(config.parent/base).resolve()
            def resolve_split(value):
                if isinstance(value,list): return None
                path=Path(value)
                if path.is_absolute():
                    # Rebase a foreign machine's absolute path only if an equivalent path exists here.
                    if path.exists(): return path
                    return base/'images'/path.name
                candidate=base/path
                if candidate.exists(): return candidate
                return config.parent/path
            train_dir=resolve_split(cfg['train']); val_dir=resolve_split(cfg['val'])
            test_dir=resolve_split(cfg['test']) if cfg.get('test') else None
            if not train_dir or not val_dir or not train_dir.exists() or not val_dir.exists(): continue
            if test_dir is None or not test_dir.exists():
                print('YOLO manifest lacks independent test split:',config)
                continue
            if not (base/'labels').exists() and not any(root.rglob('labels')): continue
            names=cfg['names']
            if isinstance(names,dict): names=[names[k] for k in sorted(names,key=lambda t:int(t))]
            else: names=list(names)
            if not names: continue
            return {'path':str(base),'train':str(train_dir),'val':str(val_dir),
                    'test':str(test_dir),'names':names}
        return None

    def convert_voc_if_available(source,dest):
        image_ext={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
        xmls=sorted(source.rglob('*.xml'))
        imgs=[p for p in source.rglob('*') if p.suffix.lower() in image_ext]
        if len(xmls)<20: return None
        from collections import defaultdict
        by_stem=defaultdict(list)
        for im in imgs: by_stem[im.stem.lower()].append(im)
        samples=[]; label_set=set()
        for xml in xmls:
            matches=by_stem.get(xml.stem.lower(),[])
            if len(matches)!=1: continue  # ambiguous or missing: do not guess
            try: tree=ET.parse(xml).getroot()
            except ET.ParseError: continue
            im=cv2.imread(str(matches[0]))
            if im is None: continue
            h,w=im.shape[:2]
            boxes=[]
            for item in tree.findall('.//object'):
                name=canonical_label(item.findtext('name',''))
                bb=item.find('bndbox')
                if not name or bb is None: continue
                try:
                    x1=float(bb.findtext('xmin')); y1=float(bb.findtext('ymin'))
                    x2=float(bb.findtext('xmax')); y2=float(bb.findtext('ymax'))
                except (TypeError,ValueError): continue
                x1=max(0.,min(x1,w-1)); y1=max(0.,min(y1,h-1))
                x2=max(0.,min(x2,w)); y2=max(0.,min(y2,h))
                if x2<=x1 or y2<=y1: continue
                boxes.append((name,x1,y1,x2,y2)); label_set.add(name)
            if boxes: samples.append((matches[0],boxes,w,h))
        if len(samples)<30 or not label_set: return None
        names=sorted(label_set); lookup={name:i for i,name in enumerate(names)}
        # Public benchmark split only. For real factory frames, replace with batch/day grouping.
        strata=[max((label for label,*_ in boxes),key=lambda z:sum(q[0]==z for q in boxes))
                 for _,boxes,_,_ in samples]
        try:
            train_s,hold_s=train_test_split(samples,test_size=.30,random_state=SEED,stratify=strata)
            val_s,test_s=train_test_split(hold_s,test_size=.50,random_state=SEED)
        except ValueError:
            train_s,hold_s=train_test_split(samples,test_size=.30,random_state=SEED)
            val_s,test_s=train_test_split(hold_s,test_size=.50,random_state=SEED)
        for split,rows in [('train',train_s),('val',val_s),('test',test_s)]:
            (dest/'images'/split).mkdir(parents=True,exist_ok=True)
            (dest/'labels'/split).mkdir(parents=True,exist_ok=True)
            for idx,(im,boxes,w,h) in enumerate(rows):
                name=f'{im.stem}_{idx:05d}'
                shutil.copy2(im,dest/'images'/split/f'{name}{im.suffix.lower()}')
                lines=[]
                for label,x1,y1,x2,y2 in boxes:
                    lines.append(f'{lookup[label]} {(x1+x2)/(2*w):.6f} {(y1+y2)/(2*h):.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}')
                (dest/'labels'/split/f'{name}.txt').write_text('\n'.join(lines))
        return {'path':str(dest),'train':'images/train','val':'images/val','test':'images/test','names':names}

    if DATA_MODE=='custom':
        DATA_YAML=Path(CUSTOM_YAML)
        assert DATA_YAML.is_file(), f'Provide custom YOLO data.yaml: {DATA_YAML}'
        custom_cfg=yaml.safe_load(DATA_YAML.read_text())
        CLASSES=list(custom_cfg['names'].values()) if isinstance(custom_cfg['names'],dict) else list(custom_cfg['names'])
        print('Using custom dataset manifest:',DATA_YAML)
    else:
        cfg=None
        # HF GC10-DET has data/images/{train,valid,test} and data/labels counterparts.
        if DATA_MODE=='gc10_hf' and (raw/'data'/'images'/'train').exists():
            orig=yaml.safe_load((raw/'data'/'data.yaml').read_text())
            names=orig.get('names')
            if isinstance(names,dict): names=[names[k] for k in sorted(names,key=lambda t:int(t))]
            if not names: raise ValueError('GC10 HF: missing class names; no assumptions allowed')
            cfg={'path':str(raw/'data'),'train':'images/train','val':'images/valid',
                 'test':'images/test','names':list(names)}
        if cfg is None: cfg=find_existing_yolo(raw)
        if cfg is None: cfg=convert_voc_if_available(raw,ROOT/f'{DATA_MODE}_converted_yolo')
        if cfg is None:
            raise ValueError(f'{DATA_MODE} downloaded but no supported labeled YOLO splits or VOC XML pairs were found. '
                             f'Inspect {raw}; use classification/segmentation adapter for non-box data.')
        DATA_YAML=ROOT/f'{DATA_MODE}_resolved.yaml'
        DATA_YAML.write_text(yaml.safe_dump(cfg,sort_keys=False))
        CLASSES=cfg['names']
        print('Resolved detection dataset:',DATA_MODE,'classes:',CLASSES)
        print('Manifest:',DATA_YAML.read_text()[:1600])


## 3. Train the known-defect detector (YOLO11n)

**Accuracy upgrades:** train with realistic exposure changes, inspect missed tiny scratches; try 960px or tiled inspection if acquisition resolution supports it; then compare YOLO11s versus YOLO11n at *equal hardware latency*. Avoid synthetic augmentations that erase the defect evidence. For deployment, benchmark **the exported INT8/ONNX model, not just `.pt`**.

In [ ]:
model=YOLO(MODEL_NAME)
train_result=model.train(data=str(DATA_YAML),epochs=EPOCHS,imgsz=IMGSZ,batch=BATCH,
                         device=DEVICE,project=str(ROOT/'runs'),name=f'{DATA_MODE}_detector',exist_ok=True,
                         patience=12,seed=SEED,workers=2,plots=True,
                         degrees=4,translate=.05,scale=.15,fliplr=.5,flipud=0.0,
                         hsv_h=0.0 if DATA_MODE in ('neu','gc10_hf','gc10') else .015,
                         hsv_s=0.0 if DATA_MODE in ('neu','gc10_hf','gc10') else .5,hsv_v=.2,
                         mosaic=.25,mixup=0.0,close_mosaic=5,verbose=False)
BEST=Path(model.trainer.best)
assert BEST.exists(), 'Training checkpoint was not created.'
detector=YOLO(str(BEST))
print('Best detector:',BEST)

In [ ]:
# Held-out test set: inspect once after selecting model on train/val.
test_metrics=detector.val(data=str(DATA_YAML),split='test',imgsz=IMGSZ,
                          batch=BATCH,device=DEVICE,plots=True,verbose=False)
print({'mAP50':round(float(test_metrics.box.map50),4),
       'mAP50_95':round(float(test_metrics.box.map),4),
       'mean_precision':round(float(test_metrics.box.mp),4),
       'mean_recall':round(float(test_metrics.box.mr),4)})
print('These are **public held-out NEU benchmark metrics**, not field acceptance or reliability claims.')

In [ ]:
# Visual sanity check. Bad localization = annotation check first, not larger model first.
if DATA_MODE=='neu':
    example_paths=list((DATA_ROOT/'images'/'test').glob('*'))[:6]
    fig,axs=plt.subplots(2,3,figsize=(14,8))
    for ax,p in zip(axs.ravel(),example_paths):
        pred=detector.predict(str(p),imgsz=IMGSZ,conf=.20,device=DEVICE,verbose=False)[0]
        ax.imshow(cv2.cvtColor(pred.plot(),cv2.COLOR_BGR2RGB)); ax.axis('off'); ax.set_title(p.name,fontsize=8)
    plt.tight_layout(); plt.show()
else:
    # The same sanity check on any supported YOLO layout.
    resolved=yaml.safe_load(Path(DATA_YAML).read_text())
    src=Path(resolved['test']) if Path(resolved['test']).is_absolute() else Path(resolved['path'])/resolved['test']
    demo_paths=[p for p in src.glob('*') if p.suffix.lower() in ('.jpg','.png','.jpeg','.bmp')][:6]
    if demo_paths:
        sample_pred=detector.predict([str(x) for x in demo_paths],imgsz=IMGSZ,device=DEVICE,verbose=False)
        fig,axs=plt.subplots(1,len(demo_paths),figsize=(3*len(demo_paths),3))
        axs=np.atleast_1d(axs)
        for ax,pred in zip(axs,sample_pred):
            ax.imshow(cv2.cvtColor(pred.plot(),cv2.COLOR_BGR2RGB)); ax.axis('off')
        plt.show()


## 4. Quantify simulated visual drift (not proof of true domain generalization)

This evaluates how *the same test items* behave after brightness, blur and noise shifts. **A real drift test still requires new factory batches/shift images held out from training.**

In [ ]:
def perturb(bgr,kind):
    if kind=='original': return bgr
    if kind=='dark': return cv2.convertScaleAbs(bgr,alpha=.6,beta=0)
    if kind=='bright': return cv2.convertScaleAbs(bgr,alpha=1.25,beta=28)
    if kind=='blur': return cv2.GaussianBlur(bgr,(7,7),1.7)
    if kind=='noise':
        rng=np.random.default_rng(7)
        return np.clip(bgr.astype(np.float32)+rng.normal(0,18,bgr.shape),0,255).astype(np.uint8)
    raise KeyError(kind)

def iou_xyxy(a,b):
    xa,ya,xb,yb=max(a[0],b[0]),max(a[1],b[1]),min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,xb-xa)*max(0,yb-ya)
    area=lambda x:max(0,x[2]-x[0])*max(0,x[3]-x[1])
    return inter/max(area(a)+area(b)-inter,1e-9)

def matched_box_recall(predictor,bgr,gt_boxes):
    pred=predictor.predict(bgr,imgsz=IMGSZ,conf=.15,device=DEVICE,verbose=False)[0]
    ps=[(int(box.cls.item()),box.xyxy[0].cpu().numpy()) for box in pred.boxes]
    used=set(); found=0
    for cls,*gt in gt_boxes:
        options=[(j,iou_xyxy(gt,pb)) for j,(pc,pb) in enumerate(ps) if j not in used and pc==cls]
        if options:
            bestj,bestiou=max(options,key=lambda z:z[1])
            if bestiou>=.5: used.add(bestj); found+=1
    return found,len(gt_boxes)

if DATA_MODE=='neu':
    stress_rows=test_rows[:min(40,len(test_rows))]
    robustness=[]
    for kind in ['original','dark','bright','blur','noise']:
        hit=total=0
        for r in stress_rows:
            bgr=cv2.imread(str(r['image']))
            h,n=matched_box_recall(detector,perturb(bgr,kind),r['boxes'])
            hit+=h; total+=n
        robustness.append({'shift':kind,'box_recall_50_proxy':round(hit/max(1,total),3),'GT_boxes':total})
    display(pd.DataFrame(robustness))
    print('Proxy = correct-class IoU≥0.5 match rate on sampled boxes; not official mAP.')

## 5. Train an unknown-anomaly branch on normal images only

The implementation is **PatchCore-inspired**, not a claim to reproduce full PatchCore: MobileNetV3 patch descriptors + unit normalization + random memory bank + cosine nearest-neighbor distances. This is small, interpretable and CPU feasible. Later compare against full PatchCore, EfficientAD and PaDiM using *actual target-product normals*. The default MVTec **tile** example is **separate from the NEU steel detector**.

To switch to your own product, put defect-free captured images in `/content/industry4_qc/my_product/normal` and set `RUN_ANOMALY_DEMO=False`. Training and calibration will use disjoint images. At least ~50–100 diverse normals makes the demo more meaningful; many more and all major shifts are needed for serious deployment.
Set `ANOMALY_SOURCE='mvtec_manual'` to use an officially downloaded MVTec dataset via your MVTec-style loader. The default `mvtec_stream_demo` preserves the previous small independent tile demo, while the registry links are printed in the dataset manager. HF Voxel51 MVTec is FiftyOne-formatted, not the classic directory layout; snapshot-download alone is not sufficient to run the classic loader.


In [ ]:
ANOM_ROOT=ROOT/'anomaly'
for sub in ('train_good','val_good','test_good','test_defect'):
    (ANOM_ROOT/sub).mkdir(parents=True,exist_ok=True)

def save_image_object(value,target):
    if isinstance(value,Image.Image): im=value.convert('RGB')
    elif isinstance(value,dict) and 'bytes' in value and value['bytes']:
        im=Image.open(io.BytesIO(value['bytes'])).convert('RGB')
    elif isinstance(value,(str,Path)): im=Image.open(value).convert('RGB')
    else: raise TypeError(f'Unexpected image field: {type(value)}')
    im.save(target)

def clear_anomaly_dir():
    for sub in ('train_good','val_good','test_good','test_defect'):
        for f in (ANOM_ROOT/sub).glob('*.png'): f.unlink()

ANOMALY_READY=False
clear_anomaly_dir()
if RUN_ANOMALY_DEMO and ANOMALY_SOURCE=='mvtec_stream_demo':
    from datasets import load_dataset
    try:
        # Specific split means we do not download all 15 MVTec categories.
        stream=load_dataset('TheoM55/mvtec_all_objects_split',split='tile.train',streaming=True)
        saved=0
        for row in stream:
            target=(ANOM_ROOT/'train_good' if saved<MAX_MVTEC_TRAIN else ANOM_ROOT/'val_good') / f'{saved:04d}.png'
            save_image_object(row['image_path'],target)
            saved+=1
            if saved >= MAX_MVTEC_TRAIN+MAX_MVTEC_VAL: break
        tests=load_dataset('TheoM55/mvtec_all_objects_split',split='tile.test',streaming=True)
        n_good=n_bad=0
        for row in tests:
            defective=int(row.get('label',0))>0
            if defective and n_bad<MAX_MVTEC_TEST_PER_CLASS:
                save_image_object(row['image_path'],ANOM_ROOT/'test_defect'/f'{n_bad:04d}.png'); n_bad+=1
            if not defective and n_good<MAX_MVTEC_TEST_PER_CLASS:
                save_image_object(row['image_path'],ANOM_ROOT/'test_good'/f'{n_good:04d}.png'); n_good+=1
            if n_good>=MAX_MVTEC_TEST_PER_CLASS and n_bad>=MAX_MVTEC_TEST_PER_CLASS: break
        print('MVTec tile images:', saved, '| test normal',n_good,'test anomaly',n_bad)
        ANOMALY_READY=(saved>=30 and n_good>=5 and n_bad>=5)
    except Exception as e:
        warnings.warn(f'MVTec mirror could not be loaded ({str(e)[:240]}). Add real normal images and set RUN_ANOMALY_DEMO=False.')
elif ANOMALY_SOURCE=='mvtec_manual':
    # MVTec's own train/good + test/good + test/<defect> format, supported by uploaded class MVTecADDataset.
    from dataset_manager import MVTecADDataset
    category_root=Path(MVTEC_MANUAL_ROOT)/MVTEC_CATEGORY
    if not category_root.exists():
        raise FileNotFoundError('Download MVTec AD from dataset_manager.py manual links and extract '+str(category_root))
    train_good_files=sorted((category_root/'train'/'good').glob('*'))
    test_good_files=sorted((category_root/'test'/'good').glob('*'))
    test_defect_files=[p for d in (category_root/'test').iterdir() if d.is_dir() and d.name!='good'
                       for p in d.iterdir() if p.suffix.lower() in ('.png','.jpg','.jpeg','.bmp')]
    if len(train_good_files)<25: raise ValueError('Not enough MVTec normal training images for split')
    rng=random.Random(SEED); rng.shuffle(train_good_files)
    n_val=max(5,len(train_good_files)//5)
    for i,p in enumerate(train_good_files):
        save_image_object(p,ANOM_ROOT/('val_good' if i<n_val else 'train_good')/f'{i:04d}.png')
    for i,p in enumerate(test_good_files): save_image_object(p,ANOM_ROOT/'test_good'/f'{i:04d}.png')
    for i,p in enumerate(test_defect_files): save_image_object(p,ANOM_ROOT/'test_defect'/f'{i:04d}.png')
    ANOMALY_READY=bool(test_good_files and test_defect_files)
    ANOMALY_PRODUCT='mvtec_'+MVTEC_CATEGORY
    # Smoke test the user's original dataset loader, including mask availability.
    ds=MVTecADDataset(str(MVTEC_MANUAL_ROOT),MVTEC_CATEGORY,split='test')
    print('MVTec loader verified:',len(ds),'labeled test images')
else:
    if NORMAL_DIR.exists():
        pths=[p for p in NORMAL_DIR.rglob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg','.bmp')]
        if len(pths)>=20:
            random.Random(SEED).shuffle(pths)
            split=max(1,int(len(pths)*.8))
            for idx,p in enumerate(pths):
                sub='train_good' if idx<split else 'val_good'
                save_image_object(str(p),ANOM_ROOT/sub/f'{idx:04d}.png')
            ANOMALY_READY=True
            ANOMALY_PRODUCT=('steel_neu' if DATA_MODE=='neu' else ('CUSTOM_PRODUCT_SET_ME' if DATA_MODE=='custom' else DATA_MODE))
            print('Own product normals:',len(pths),'(no held-out defect test until you provide it)')
        else: print('Need at least 20 normal images; anomaly step disabled')
    else: print('No own normal image folder; anomaly step disabled')
print('Anomaly branch ready:',ANOMALY_READY)

In [ ]:
import torchvision
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
import torch.nn as nn
import torch.nn.functional as F

# Intermediate feature resolution retains spatial detail better than pooled classification logits.
weights=MobileNet_V3_Small_Weights.DEFAULT
backbone=mobilenet_v3_small(weights=weights).eval()
encoder=nn.Sequential(*list(backbone.features.children())[:8]).eval().to(DEVICE)
preprocess=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),
                               transforms.Normalize(mean=[.485,.456,.406],std=[.229,.224,.225])])

def encode_good_image(path):
    x=preprocess(Image.open(path).convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.inference_mode(): fm=encoder(x).float().cpu().numpy()[0]
    c,h,w=fm.shape
    descriptors=fm.reshape(c,h*w).T.astype(np.float32)
    descriptors/=np.maximum(np.linalg.norm(descriptors,axis=1,keepdims=True),1e-8)
    return descriptors,(h,w)

MEMORY_PATCHES=1800  # for CPU, test bank sizes 300, 800, 1800: accuracy vs latency
MEMORY_BANK_PATH=ROOT/'normal_bank.npz'
ENCODER_ONNX=ROOT/'mobilenet_patch_encoder.onnx'

if ANOMALY_READY:
    train_good=sorted((ANOM_ROOT/'train_good').glob('*.png'))
    train_patches=[encode_good_image(p)[0] for p in train_good]
    stack=np.concatenate(train_patches,axis=0)
    rng=np.random.default_rng(SEED)
    memory_bank=stack[rng.choice(len(stack),size=min(len(stack),MEMORY_PATCHES),replace=False)]
    del train_patches,stack; gc.collect()
    print('Memory bank shape:',memory_bank.shape, '| MB:',memory_bank.nbytes/1e6)
else:
    memory_bank=None

In [ ]:
# Distance-only anomaly head; threshold is calibrated ONLY on separate defect-free images.
def anomaly_score(path):
    desc,(H,W)=encode_good_image(path)
    nearest=1.0-(desc @ memory_bank.T).max(axis=1)
    tail=max(1,math.ceil(len(nearest)*.01))
    score=float(np.mean(np.sort(nearest)[-tail:]))
    return score,nearest.reshape(H,W)

if ANOMALY_READY:
    good_cal=sorted((ANOM_ROOT/'val_good').glob('*.png'))
    cal_scores=np.array([anomaly_score(p)[0] for p in good_cal])
    # 99th quantile is a target calibration strategy; small n => noisy false alarm control.
    ANOMALY_THRESHOLD=float(np.quantile(cal_scores,.99))
    np.savez_compressed(MEMORY_BANK_PATH,bank=memory_bank.astype(np.float32),
                        threshold=np.array(ANOMALY_THRESHOLD,dtype=np.float32),
                        domain=np.array(ANOMALY_PRODUCT))
    print('Threshold (held-out good 99th percentile):',ANOMALY_THRESHOLD)
    print('Validation normal count:',len(cal_scores),' — 99th percentile is NOT guaranteed 1% field FPR.')

In [ ]:
if ANOMALY_READY:
    y_true=[]; y_score=[]
    for label,sub in ((0,'test_good'),(1,'test_defect')):
        for p in sorted((ANOM_ROOT/sub).glob('*.png')):
            y_true.append(label); y_score.append(anomaly_score(p)[0])
    if len(set(y_true))==2:
        auc=roc_auc_score(y_true,y_score)
        flagged=np.array(y_score)>ANOMALY_THRESHOLD
        tn,fp,fn,tp=confusion_matrix(y_true,flagged,labels=[0,1]).ravel()
        print({'MVTec_tile_image_AUROC':round(float(auc),4),
               'normal_false_alarm_rate':round(float(fp/max(fp+tn,1)),4),
               'defect_sensitivity':round(float(tp/max(tp+fn,1)),4),
               'normal_test':tn+fp,'defect_test':tp+fn})
        fig,ax=plt.subplots(figsize=(7,3))
        ax.hist(np.array(y_score)[np.array(y_true)==0],bins=18,alpha=.65,label='Test good')
        ax.hist(np.array(y_score)[np.array(y_true)==1],bins=18,alpha=.65,label='Test anomalous')
        ax.axvline(ANOMALY_THRESHOLD,color='black',ls='--',label='Calibrated threshold')
        ax.legend(); ax.set_xlabel('Nearest-neighbor anomaly score'); plt.show()
    else: print('No labeled anomalies available: report normal-score distribution only, no AUROC.')

In [ ]:
if ANOMALY_READY:
    examples=(sorted((ANOM_ROOT/'test_good').glob('*.png'))[:2] +
              sorted((ANOM_ROOT/'test_defect').glob('*.png'))[:2])
    fig,axs=plt.subplots(len(examples),2,figsize=(9,3*len(examples)))
    for row,p in enumerate(examples):
        val,heat=anomaly_score(p)
        rgb=np.asarray(Image.open(p).convert('RGB'))
        axs[row,0].imshow(rgb); axs[row,0].set_title(f'{p.parent.name} • {val:.3f}'); axs[row,0].axis('off')
        axs[row,1].imshow(rgb); axs[row,1].imshow(cv2.resize(heat,(rgb.shape[1],rgb.shape[0])),cmap='inferno',alpha=.55)
        axs[row,1].set_title('Anomaly localization heatmap (approximate)'); axs[row,1].axis('off')
    plt.tight_layout(); plt.show()

## 6. Rule-based severity, abstention and local SOP

A bounding box has **area**, not crack depth, material strength or pharmacological safety. Severity here is a transparent **prototype policy**, not a trained safety classifier. Confirm severity using manufacturing tolerances, critical regions, and expert-approved examples. Avoid letting an LLM issue automatic machine-stop commands.

In [ ]:
def inspect_quality(bgr):
    gray=cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY)
    brightness=float(gray.mean()); blur=float(cv2.Laplacian(gray,cv2.CV_64F).var())
    problems=[]
    if brightness<25: problems.append('underexposed')
    if brightness>240: problems.append('overexposed')
    if blur<18: problems.append('blurry_or_low_texture')
    return {'brightness':round(brightness,1),'sharpness':round(blur,1),
            'passed':len(problems)==0,'problems':problems}

# These are *illustrative* prototype priorities requiring plant QA sign-off.
SEVERITY_BASE={'crazing':'major','rolled-in_scale':'major','inclusion':'major',
               'scratches':'minor','pitted_surface':'minor','patches':'minor'}
SOPS={
    'crazing':'Hold item; investigate surface crack and material/rolling stress.',
    'rolled-in_scale':'Hold item; inspect descaling process and rollers.',
    'inclusion':'Hold item; inspect material batch and foreign particle origin.',
    'scratches':'Check fixture, roller, and handling contact points.',
    'pitted_surface':'Inspect corrosion prevention and surface finishing.',
    'patches':'Check surface contamination and coating/process uniformity.',
    'unknown_anomaly':'Isolate item for expert inspection; retain evidence.',
    'image_quality':'Recapture with controlled illumination and stable focus.'
}

def classify_prototype_severity(name,area_ratio):
    base=SEVERITY_BASE.get(name,'major')
    if name=='crazing' and area_ratio>.12: return 'critical_candidate'
    if base=='minor' and area_ratio>.15: return 'major_candidate'
    return base+'_candidate'

def fusion_decision(known,quality,anom_score=None,anom_threshold=None,same_product=False):
    if not quality['passed']:
        return 'RECAPTURE','Image quality gate failed'
    if known:
        return 'HOLD','Recognized surface defect(s); manual QC confirmation required'
    if same_product and anom_score is not None and anom_threshold is not None:
        if anom_score>anom_threshold: return 'REVIEW','Unclassified anomaly'
        return 'ACCEPT_CANDIDATE','No anomaly detected; subject to validated acceptance criteria'
    return 'REVIEW_UNVERIFIED','No matched product-specific normal model; cannot certify clean'

def make_event(img_bgr, product='steel_neu', include_anomaly=False, conf=.25):
    quality=inspect_quality(img_bgr)
    boxes=[]
    if quality['passed']:
        pred=detector.predict(img_bgr,device=DEVICE,conf=conf,imgsz=IMGSZ,verbose=False)[0]
        h,w=img_bgr.shape[:2]
        for b in pred.boxes:
            cls=detector.names[int(b.cls.item())]
            x1,y1,x2,y2=b.xyxy[0].cpu().tolist()
            area_ratio=max(0,x2-x1)*max(0,y2-y1)/(w*h)
            boxes.append({'class':cls,'confidence':round(float(b.conf.item()),3),
                          'bbox_xyxy':[round(x1),round(y1),round(x2),round(y2)],
                          'bbox_area_ratio':round(area_ratio,4),
                          'severity_estimate':classify_prototype_severity(cls,area_ratio)})
    # CRITICAL anti-leakage: anomaly score only if memory bank is the *same* product.
    same_product=(include_anomaly and ANOMALY_READY and product==ANOMALY_PRODUCT)
    anomaly=None
    if same_product and quality['passed']:
        with Image.fromarray(cv2.cvtColor(img_bgr,cv2.COLOR_BGR2RGB)) as im:
            tmp=ROOT/'_inspection_tmp.png'; im.save(tmp)
        anomaly=float(anomaly_score(tmp)[0]); tmp.unlink(missing_ok=True)
    decision,reason=fusion_decision(boxes,quality,anomaly,
                                   ANOMALY_THRESHOLD if same_product else None,same_product)
    defect_type=boxes[0]['class'] if boxes else ('unknown_anomaly' if decision=='REVIEW' else 'none')
    sop=SOPS.get(defect_type,'Continue documented inspection procedure.')
    return {'utc_time':pd.Timestamp.utcnow().isoformat(),'product_id':product,
            'quality':quality,'defects':boxes,'anomaly_score':anomaly,
            'decision':decision,'decision_reason':reason,
            'sop_recommendation':sop,'model_file':str(BEST)}

if DATA_MODE=='neu':
    demo_img=cv2.imread(str(test_rows[0]['image']))
    example_event=make_event(demo_img,'steel_neu',include_anomaly=False)
    print(json.dumps(example_event,indent=2,default=str))

## 7. Drift and active learning — sample selection, not unsafe automatic retraining

For each manufacturing recipe keep a **validated reference distribution** (brightness, sharpness, embeddings, good-unit anomaly scores, detector confidence, operator override rate). Test drift per **day, shift, batch and camera**; a warning **triggers image-quality investigation / sampling**, not an immediate update to the normal memory bank. Only QA-confirmed good images can be proposed for later retraining.

In [ ]:
from scipy.stats import ks_2samp

def image_stats(path):
    bgr=cv2.imread(str(path))
    if bgr is None: return None
    g=cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY)
    return {'brightness':float(g.mean()),'contrast':float(g.std()),
            'sharpness':float(cv2.Laplacian(g,cv2.CV_64F).var())}

# This is an illustrative shifted-image monitoring example; groups are correlated.
if DATA_MODE=='neu':
    candidates=test_rows[:min(40,len(test_rows))]
    baseline=pd.DataFrame([image_stats(r['image']) for r in candidates])
    shifted=[]
    for r in candidates:
        bgr=perturb(cv2.imread(str(r['image'])),'dark')
        g=cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY)
        shifted.append({'brightness':float(g.mean()),'contrast':float(g.std()),
                        'sharpness':float(cv2.Laplacian(g,cv2.CV_64F).var())})
    altered=pd.DataFrame(shifted)
    results=[]
    for col in baseline.columns:
        ks,p=ks_2samp(baseline[col],altered[col]); results.append({
            'feature':col,'reference_mean':round(baseline[col].mean(),2),
            'shifted_mean':round(altered[col].mean(),2),'KS_distance':round(ks,3),
            'p_value_EXPLORATORY':round(p,5)})
    display(pd.DataFrame(results))
    print('Do not interpret iid KS p-values as proof of drift on serially correlated factory video.')

# A local review queue saves conflicts/low confidence for labeling.
REVIEW_CSV=ROOT/'review_queue.csv'
def queue_for_review(event,image_path,reason):
    line={'utc_time':pd.Timestamp.utcnow().isoformat(),'image_path':str(image_path),
          'reason':reason,'product_id':event['product_id'],
          'decision':event['decision'],'operator_label':'PENDING','model_version':str(BEST)}
    pd.DataFrame([line]).to_csv(REVIEW_CSV,mode='a',header=not REVIEW_CSV.exists(),index=False)
    return line
if DATA_MODE=='neu':
    if example_event['decision']!='ACCEPT_CANDIDATE':
        queue_for_review(example_event,test_rows[0]['image'],'manual_demo_review')
    print('Review CSV:',REVIEW_CSV)

## 8. Optional: supervised crop/severity classifier (ONLY if you have expert-labeled samples)

To avoid **inventing severity labels from image class**, this is disabled by default. Create `/content/severity_crops/train/{minor,major,critical}/...` and matching `/val/...` with expert-reviewed crops; re-run with the flag. Report confusion and *critical-class recall*. The inference integration must use an actual trained checkpoint, not this placeholder.

In [ ]:
RUN_SEVERITY_TRAIN=False
SEVERITY_ROOT=Path('/content/severity_crops')
if RUN_SEVERITY_TRAIN:
    from torch.utils.data import DataLoader
    from torchvision.datasets import ImageFolder
    train_aug=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(.5),
                                  transforms.ColorJitter(brightness=.15,contrast=.15),
                                  transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    val_aug=preprocess
    tr=ImageFolder(SEVERITY_ROOT/'train',transform=train_aug)
    va=ImageFolder(SEVERITY_ROOT/'val',transform=val_aug)
    assert tr.classes==va.classes, 'Train and validation severity class mappings must match'
    mdl=mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
    mdl.classifier[-1]=nn.Linear(mdl.classifier[-1].in_features,len(tr.classes))
    mdl=mdl.to(DEVICE)
    opt=torch.optim.AdamW(mdl.parameters(),lr=2e-4,weight_decay=.01)
    for epoch in range(10):
        mdl.train(); loss_sum=0
        for x,y in DataLoader(tr,batch_size=24,shuffle=True,num_workers=2):
            x,y=x.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(); loss=F.cross_entropy(mdl(x),y)
            loss.backward(); opt.step(); loss_sum+=float(loss.item())
        print('epoch',epoch+1,'mean_batch_loss',round(loss_sum/max(1,len(tr)),4))
    mdl.eval(); gold=[];guess=[]
    with torch.inference_mode():
        for x,y in DataLoader(va,batch_size=24):
            pred=mdl(x.to(DEVICE)).argmax(1).cpu()
            gold.extend(y.tolist()); guess.extend(pred.tolist())
    print(classification_report(gold,guess,target_names=tr.classes,zero_division=0))
    torch.save({'state_dict':mdl.cpu().state_dict(),'class_names':tr.classes},ROOT/'severity_mobilenet.pt')
else:
    print('Severity classifier correctly disabled: no real expert severity labels provided.')

## Optional separate classification task — magnetic tile (from your registry)

Set `CLASSIFICATION_SOURCE='magnetic_tile'` in the configuration cell to download via `dataset_manager.py` and train an **image classifier**. Unlike box-based detection, a class-folder image classifier **cannot draw defect boxes**. It uses a separate MobileNet-style/YOLO classification head and a separate accuracy report. Keep this model separate from the steel/PCB detector; image-level labels are not fabricated into bounding boxes.

The code searches for a folder with per-class subdirectories. If your Kaggle mirror is organized differently (e.g., with image/mask pairs), it deliberately stops for a manual adapter. Original `organize_classification_dataset` is used to make train/val/test folders; **avoid using this random split for correlated factory parts**.

In [ ]:
if CLASSIFICATION_SOURCE=='magnetic_tile':
    cls_raw=download_from_registry(CLASSIFICATION_SOURCE)
    def locate_class_folders(raw):
        candidates=[]
        for parent in [raw]+[p for p in raw.rglob('*') if p.is_dir()]:
            dirs=[p for p in parent.iterdir() if p.is_dir()]
            valid=[]
            for folder in dirs:
                ims=[p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in dm.IMAGE_EXTENSIONS
                     and 'mask' not in p.stem.lower()]
                if len(ims)>=4: valid.append(folder)
            if len(valid)>=2: candidates.append((sum(len(list(d.glob('*'))) for d in valid),parent,valid))
        if not candidates: raise ValueError('No flat class-folder layout detected. Inspect folder and write a manual adapter, do not turn masks into classes.')
        _,parent,dirs=max(candidates,key=lambda z:z[0]); return parent,dirs
    class_root,cls_dirs=locate_class_folders(cls_raw)
    print('Detected classification classes:',[x.name for x in cls_dirs])
    from PIL import Image as PILImage
    cls_clean=ROOT/'classification_clean'
    for d in cls_dirs:
        dest=cls_clean/d.name;dest.mkdir(parents=True,exist_ok=True)
        for p in d.iterdir():
            if p.suffix.lower() not in dm.IMAGE_EXTENSIONS or 'mask' in p.stem.lower(): continue
            try:
                with PILImage.open(p) as im: im.verify()
            except Exception: continue
            shutil.copy2(p,dest/p.name)
    cls_processed=ROOT/'data'/'processed'/'classification'/'magnetic_tile'
    dm.organize_classification_dataset(cls_clean,cls_processed,seed=SEED)
    class_model=YOLO('yolo11n-cls.pt')
    class_model.train(data=str(cls_processed),epochs=CLASSIFICATION_EPOCHS,imgsz=224,
                      project=str(ROOT/'runs'),name='magnetic_tile_classification',
                      batch=BATCH,seed=SEED,device=DEVICE,exist_ok=True)
    cls_metrics=class_model.val(data=str(cls_processed),split='test',device=DEVICE)
    print('Classification test metrics:',cls_metrics.results_dict)
    # An independently exported classifier; the standard edge_inspector does NOT run it.
    cls_best=Path(class_model.trainer.best)
    classifier_onnx=Path(YOLO(str(cls_best)).export(format='onnx',imgsz=224,opset=12,simplify=True))
    shutil.copy2(classifier_onnx,ROOT/'magnetic_tile_classifier.onnx')
    print('Optional classification ONNX:',ROOT/'magnetic_tile_classifier.onnx')
else:
    print('Classification route disabled; set CLASSIFICATION_SOURCE=magnetic_tile to train it.')

## 9. Export ONNX artifacts for Raspberry Pi and compare runtimes

Vision models **do not use ExecuTorch** in this reference package. CPU option: YOLO ONNX / ONNXRuntime. Hailo option: convert trained detector into **hardware-specific HEF** on a supported Linux x86-64 host, then run with HailoRT. Llama 3.2 1B runs via **ExecuTorch on Raspberry Pi Arm CPU**, as a separate event-triggered process.

**For Hailo:** use `YOLO('best.pt').export(format='hailo',name='hailo8l',data='data.yaml',imgsz=640)` on a compatible Linux x86-64 host with the correct Hailo Dataflow Compiler. A Colab T4 is *not* a substitute for compiling HEF with that compiler.

In [ ]:
# CPU ONNX export; compare ONNX and PyTorch predictions/metrics before deployment.
YOLO_ONNX=Path(detector.export(format='onnx',imgsz=IMGSZ,opset=12,simplify=True,dynamic=False))
print('YOLO ONNX:',YOLO_ONNX,'MB:',round(YOLO_ONNX.stat().st_size/1e6,1))
if ANOMALY_READY:
    encoder.cpu().eval()
    dummy=torch.randn(1,3,224,224)
    torch.onnx.export(encoder,dummy,str(ENCODER_ONNX),opset_version=17,
                      input_names=['image'],output_names=['features'],
                      do_constant_folding=True)
    print('Anomaly encoder ONNX:',ENCODER_ONNX,'MB:',round(ENCODER_ONNX.stat().st_size/1e6,1))
    encoder.to(DEVICE)
print('NOTE: ONNX export is NOT INT8 quantization. Quantize and validate on actual Pi/Hailo separately.')

In [ ]:
# Check ONNX has similar detections on a few fixed test images.
onnx_model=YOLO(str(YOLO_ONNX),task='detect')
if DATA_MODE=='neu':
    images_check=[str(x['image']) for x in test_rows[:min(8,len(test_rows))]]
    disagreements=[]
    for source in images_check:
        a=detector.predict(source,imgsz=IMGSZ,conf=.25,verbose=False,device=DEVICE)[0]
        b=onnx_model.predict(source,imgsz=IMGSZ,conf=.25,verbose=False,device='cpu')[0]
        disagreements.append({'image':Path(source).name,'torch_boxes':len(a.boxes),'onnx_boxes':len(b.boxes)})
    display(pd.DataFrame(disagreements))
    print('Box counts are a smoke check, NOT a complete quantization/parity validation.')

## 10. Llama operator advisory contract (ExecuTorch on Pi)

Do not ask a language model to classify pixels or override QC decisions. Llama receives **structured facts + retrieved local SOP** and supplies a human-readable checklist. Model files are often license-gated and the Pi runner must be built/tested separately. This cell produces a prompt to call from the edge application; it does not falsely claim local Llama was executed in Colab.

In [ ]:
def llama_payload(event):
    return {'role':'quality_control_advisor','must_not_override':event['decision'],
            'inspection':{'product_id':event['product_id'],
                          'known_defects':event['defects'][:5],
                          'anomaly_score':event['anomaly_score'],
                          'quality':event['quality'],'decision':event['decision']},
            'retrieved_approved_sop':event['sop_recommendation'],
            'instructions':['Ground checklist ONLY in input and approved SOP.',
                            'State suspected causes as possibilities, never certainties.',
                            'Never instruct autonomous actuation; human signs off.',
                            'Do not change HOLD/RECAPTURE/REVIEW state.',
                            'Return concise checklist and urgency.']}
if DATA_MODE=='neu':
    prompt=json.dumps(llama_payload(example_event),indent=2)
    (ROOT/'sample_llama_prompt.json').write_text(prompt)
    print(prompt[:2600])
    print('This is prompt generation only. The required Llama inference happens with ExecuTorch on the Pi.')

## 11. Build offline model manifest, save experiment metadata and bundle

For safety, this manifest **marks steel and tile as different product profiles**. A factory integration must set product-specific thresholds based on new target normal and defect data and then conduct a shadow-mode validation. The demo pipeline **does not** contain trained criticality measurements, segmentation masks, or a certified pass/fail decision.

In [ ]:
manifest={
 'project':'Industry4_Offline_QC','seed':SEED,'vision_model':'YOLO11n',
 'vision_model_onnx':YOLO_ONNX.name,'vision_classes':CLASSES,
 'vision_product':'steel_neu' if DATA_MODE=='neu' else ('CUSTOM_PRODUCT_SET_ME' if DATA_MODE=='custom' else DATA_MODE),
 'anomaly_model':'MobileNetV3 patch nearest-neighbor (PatchCore-inspired)' if ANOMALY_READY else None,
 'anomaly_product':ANOMALY_PRODUCT if ANOMALY_READY else None,
 'anomaly_encoder_onnx':ENCODER_ONNX.name if ANOMALY_READY else None,
 'anomaly_bank':MEMORY_BANK_PATH.name if ANOMALY_READY else None,
 'anomaly_threshold':float(ANOMALY_THRESHOLD) if ANOMALY_READY else None,
 'advisory_model':'Llama 3.2 1B Instruct via ExecuTorch Arm CPU, deployed separately',
 'factory_certified':False,
 'classifier_onnx':'magnetic_tile_classifier.onnx' if (ROOT/'magnetic_tile_classifier.onnx').exists() else None,
 'dataset_source':DATA_MODE,'registered_dataset':dm.DATASETS.get(DATA_MODE,{}).get('id'),
 'limitations':['public dataset results do not prove factory drift robustness','anomaly bank must match product and camera',
                'prototype severity rules are not expert-certified','actual field drift/batch holdout not yet evaluated']}
(ROOT/'manifest.json').write_text(json.dumps(manifest,indent=2))
shutil.copy2(YOLO_ONNX,ROOT/'yolo_detector.onnx')
shutil.copy2(BEST,ROOT/'yolo_detector_best.pt')
print(json.dumps(manifest,indent=2))

## 12. Next experiments that have the biggest impact on real factory performance

1. **Acquisition before architecture:** fixed lens/focus, synchronized illumination and exposure, calibration target, unique part ID. For shiny metal consider diffuse/dark-field or polarized lights; ensure cracks remain visible in the chosen lighting.
2. **Real data:** 300–1,000+ confirmed normal images across batches/shift/temperature, plus *every obtainable known defect* with boxes and ideally pixel masks; deduplicate near-identical frames and isolate a truly future/batch-held-out test.
3. **Model comparison:** YOLO11n at 640 vs tiled 960/1024 vs YOLO11s; PatchCore-inspired vs full PatchCore vs EfficientAD. Choose the **Pareto frontier** of product-level miss rate versus latency and false alarms.
4. **Calibration:** use defect-free *held-out* target images to choose FPR target, then independently evaluate sensitivity. Confidence alone is not probability. Add class-specific thresholds, local critical ROI, and 'manual review' reject option.
5. **Advanced:** hard negative mining from false alerts; targeted photometric/domain randomization; batch-wise **worst-group recall**; knowledge distillation if teacher beats student; defect-preserving tiling; risk-aware selective prediction; temporal same-part voting; local SOP retrieval.
6. **Continuous improvement:** drift alarms only trigger diagnosis and QA sampling. Require QA approval before new normal samples enter memory or any model is redeployed. Version every artifact and keep the rollback path.
7. **Edge verification:** profile **camera→decision p50/p95**, false alarms per 1,000 normal units, miss rate by defect/size/lot, temperature throttling, memory, offline/no-network operation and human advisory latency. Llama can be delayed; the rule-based alert must be immediate.

**Scientific warning:** conformity and risk guarantees cannot be inferred from tiny datasets. For a measured one-sided false-positive-rate limit and conformal coverage, use a proper exchangeability-aware calibration protocol and report actual error bars with batch-level validation.

### Documentation links and exact edge responsibilities

- NEU-DET dataset: https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database
- MVTec AD dataset/licensing: https://www.mvtec.com/research-teaching/datasets/mvtec-ad
- Ultralytics YOLO Python: https://docs.ultralytics.com/usage/python/
- Ultralytics Hailo export: https://docs.ultralytics.com/integrations/hailo/
- Raspberry Pi AI HAT docs: https://www.raspberrypi.com/documentation/computers/ai.html
- ExecuTorch Raspberry Pi tutorial: https://docs.pytorch.org/executorch/stable/raspberry_pi_llama_tutorial.html
- ExecuTorch model example: https://github.com/pytorch/executorch/blob/main/examples/models/llama/README.md

**Separation of runtimes:** `YOLO .onnx → ONNXRuntime on CPU` **OR** `.hef → HailoRT on NPU`; `Llama .pte → ExecuTorch llama_main on Arm CPU`. The AI HAT+ and AI HAT+ 2 require target-specific vision exports; do not treat ONNX itself as an Hailo executable.

In [ ]:
%%writefile /content/industry4_qc/edge_inspector.py
#!/usr/bin/env python3
"""Offline Raspberry Pi / Arm edge reference app.

Vision: YOLO ONNX via Ultralytics+ONNXRuntime on CPU (or replace with a HailoRT HEF adapter).
Anomaly: MobileNetV3 feature ONNX + local cosine nearest-neighbour memory bank.
Advisory: local SOP immediately; optional ExecuTorch llama_main on Arm CPU asynchronously.
Do NOT run an MVTec tile anomaly bank against steel, PCB, or another product domain.
"""
import argparse
import concurrent.futures
import hashlib
import json
import sqlite3
import subprocess
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

SOPS = {
    'crazing': ('HOLD', 'Inspect cracks with magnification; check forming stress and material batch.', 'high'),
    'inclusion': ('HOLD', 'Check incoming material quality and surface cleanliness.', 'medium'),
    'patches': ('REVIEW', 'Check coating consistency, contamination, and illumination.', 'medium'),
    'pitted_surface': ('HOLD', 'Review corrosion protection and surface-treatment process.', 'medium'),
    'rolled-in_scale': ('HOLD', 'Check scale removal and rolling process settings.', 'high'),
    'scratches': ('HOLD', 'Inspect handling, rollers, fixtures, and contact surfaces.', 'medium'),
    'unknown_anomaly': ('REVIEW', 'Isolate item and inspect the highlighted area manually.', 'high'),
    'image_quality': ('RECAPTURE', 'Clean lens, stabilize camera, or correct illumination.', 'medium'),
    'ok': ('CONTINUE', 'No defect signal identified; keep periodic manual audits.', 'low'),
}


def stable_hash(path):
    sha = hashlib.sha256()
    with open(path, 'rb') as fh:
        for part in iter(lambda: fh.read(1024 * 1024), b''):
            sha.update(part)
    return sha.hexdigest()[:16]


def image_quality(image, blur_min=18., dark_min=25., bright_max=240.):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    brightness = float(gray.mean())
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    # Tunable demonstration thresholds, NOT certified factory thresholds.
    flags = []
    if brightness < dark_min: flags.append('underexposed')
    if brightness > bright_max: flags.append('overexposed')
    if sharpness < blur_min: flags.append('blurry_or_low_texture')
    return {'mean_brightness': brightness, 'laplacian_variance': sharpness,
            'flags': flags, 'passed': not flags}


class OptionalAnomaly:
    def __init__(self, onnx_path, bank_path):
        import onnxruntime as ort
        self.session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
        data = np.load(bank_path, allow_pickle=False)
        self.bank = data['bank'].astype(np.float32)
        self.threshold = float(data['threshold'])
        self.domain = str(data['domain'].item())
        self.input_name = self.session.get_inputs()[0].name

    def score(self, bgr):
        rgb = cv2.cvtColor(cv2.resize(bgr, (224, 224)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
        x = (rgb - np.array([.485, .456, .406], np.float32)) / np.array([.229, .224, .225], np.float32)
        tensor = x.transpose(2, 0, 1)[None].astype(np.float32)
        fmap = self.session.run(None, {self.input_name: tensor})[0][0]
        feats = fmap.reshape(fmap.shape[0], -1).T
        feats = feats / np.maximum(np.linalg.norm(feats, axis=1, keepdims=True), 1e-9)
        # Cosine distance to nearest approved-normal descriptor.
        distances = 1. - np.max(feats @ self.bank.T, axis=1)
        tail = max(1, int(np.ceil(len(distances) * .01)))
        score = float(np.mean(np.sort(distances)[-tail:]))
        heat = distances.reshape(fmap.shape[1], fmap.shape[2])
        heat = cv2.resize(heat, (bgr.shape[1], bgr.shape[0]), interpolation=cv2.INTER_LINEAR)
        return score, heat


def make_advice(event):
    kind = event['defects'][0]['class'] if event['defects'] else (
        'unknown_anomaly' if event['anomaly_alarm'] else 'ok')
    if event['decision'] == 'RECAPTURE': kind = 'image_quality'
    action, step, urgency = SOPS.get(kind, SOPS['unknown_anomaly'])
    return {'recommended_action': action, 'urgency': urgency,
            'possible_cause': 'Cause not confirmed; verify on the production line.',
            'operator_checklist': [step, 'Record operator confirmation or override.'],
            'source': 'approved_local_sop_template'}


def choose_decision(quality, defects, anomaly_alarm, anomaly_enabled):
    if not quality['passed']: return 'RECAPTURE'
    # Conservative safety policy: no clean-product evidence = no automatic acceptance.
    if any(d['class'] in ('crazing', 'rolled-in_scale') for d in defects): return 'HOLD'
    if defects: return 'HOLD'
    if anomaly_alarm: return 'REVIEW'
    if not anomaly_enabled: return 'REVIEW_UNVERIFIED'
    return 'ACCEPT_CANDIDATE'   # only after target-domain validation is complete


def make_prompt(event, sop):
    payload = {'inspection': {
        'product_id': event['product_id'], 'decision': event['decision'],
        'defects': event['defects'], 'anomaly_score': event['anomaly_score'],
        'image_quality': event['quality']}, 'approved_sop': sop}
    return ('You are an offline manufacturing QC advisory assistant. '
            'Only use the JSON facts and local SOP. Never reverse HOLD or RECAPTURE. '
            'Do not claim defect depth, physical measurements or root cause certainty. '
            'Give a concise operator checklist. Never instruct automatic machine control. '
            'Input: ' + json.dumps(payload, separators=(',', ':')))


def call_executorch(prompt, runner, model, tokenizer):
    cmd = [str(runner), '--model_path', str(model), '--tokenizer_path', str(tokenizer),
           '--seq_len', '512', '--prompt', prompt[:2000]]
    completed = subprocess.run(cmd, capture_output=True, text=True, timeout=180, check=False)
    if completed.returncode != 0:
        raise RuntimeError((completed.stderr or completed.stdout)[-800:])
    return completed.stdout.strip()[-4000:]


def open_db(path):
    db = sqlite3.connect(path, check_same_thread=False)
    db.execute('''CREATE TABLE IF NOT EXISTS inspections (
        event_id TEXT PRIMARY KEY, timestamp_utc TEXT, product_id TEXT, decision TEXT,
        defect_json TEXT, anomaly_score REAL, quality_json TEXT, model_sha TEXT,
        vision_ms REAL, sop_json TEXT, llama_output TEXT, operator_feedback TEXT)''')
    db.commit()
    return db


def main():
    p = argparse.ArgumentParser(description='Industry 4.0 offline edge inspection')
    p.add_argument('--weights', help='Trained YOLO ONNX file (CPU path)')
    p.add_argument('--detector-product', default='steel_neu', help='Product on which this detector was trained')
    p.add_argument('--disable-detector', action='store_true', help='Run only target-domain anomaly branch')
    p.add_argument('--source', default='0', help='Camera integer, video path, or image path')
    p.add_argument('--product', default='steel_neu', help='Product/recipe identifier')
    p.add_argument('--anomaly-encoder', type=Path)
    p.add_argument('--anomaly-bank', type=Path)
    p.add_argument('--llama-runner', type=Path)
    p.add_argument('--llama-model', type=Path)
    p.add_argument('--llama-tokenizer', type=Path)
    p.add_argument('--db', type=Path, default=Path('inspections.sqlite3'))
    p.add_argument('--interval', type=float, default=.7, help='Minimum seconds per inspection')
    p.add_argument('--conf', type=float, default=.25)
    p.add_argument('--view', action='store_true', help='Requires desktop display')
    p.add_argument('--save-images', type=Path, help='Optional local evidence directory')
    args = p.parse_args()

    detector = None
    checksum = 'none'
    if not args.disable_detector:
        if not args.weights: raise ValueError('Provide --weights or use --disable-detector')
        if args.detector_product != args.product:
            raise ValueError('Detector product domain differs from --product. Do not run cross-domain detector.')
        weights = Path(args.weights)
        if not weights.is_file(): raise FileNotFoundError(weights)
        detector = YOLO(str(weights), task='detect')
        checksum = stable_hash(weights)
    anomaly = None
    if args.anomaly_encoder and args.anomaly_bank:
        anomaly = OptionalAnomaly(args.anomaly_encoder, args.anomaly_bank)
        if anomaly.domain != args.product:
            raise ValueError(f'Anomaly bank trained for {anomaly.domain}; requested {args.product}. '
                             'Never fuse incompatible product domains.')
    wants_llama = [args.llama_runner, args.llama_model, args.llama_tokenizer]
    llama_ready = all(x and x.is_file() for x in wants_llama)
    db = open_db(args.db)
    if args.save_images: args.save_images.mkdir(parents=True, exist_ok=True)

    pool = concurrent.futures.ThreadPoolExecutor(max_workers=1) if llama_ready else None
    pending = None

    def advisory_job(event_id, prompt):
        try:
            output = call_executorch(prompt, *wants_llama)
        except Exception as exc:
            output = 'EXECUTORCH_ERROR: ' + str(exc)
        # A separate SQLite connection is used because worker is on a different thread.
        with sqlite3.connect(args.db) as worker_db:
            worker_db.execute('UPDATE inspections SET llama_output=? WHERE event_id=?',
                              (output, event_id))
        print('Llama advice [not machine-control input]:', output[:500])

    src = int(args.source) if args.source.isdigit() else args.source
    image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')
    single_image = isinstance(src, str) and Path(src).suffix.lower() in image_extensions
    capture = None if single_image else cv2.VideoCapture(src)
    if not single_image and not capture.isOpened(): raise RuntimeError(f'Camera/video unavailable: {src}')
    last_time = -1e9
    try:
        while True:
            if single_image:
                frame = cv2.imread(str(src))
                if frame is None: raise RuntimeError(f'Unable to load image {src}')
            else:
                okay, frame = capture.read()
                if not okay: break
            if time.monotonic() - last_time < args.interval:
                if single_image: break
                continue
            last_time = time.monotonic()
            start = time.perf_counter()
            quality = image_quality(frame)
            # For invalid frames: do not call image clean. Request recapture.
            result = detector.predict(frame, device='cpu', imgsz=640, conf=args.conf,
                                      verbose=False)[0] if quality['passed'] and detector is not None else None
            defects = []
            if result is not None:
                for box in result.boxes:
                    idx = int(box.cls.item())
                    x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
                    defects.append({'class': str(detector.names[idx]),
                                    'confidence': round(float(box.conf.item()), 4),
                                    'bbox_xyxy': [round(v, 1) for v in (x1, y1, x2, y2)],
                                    'area_ratio': round(max(0, x2-x1)*max(0, y2-y1)/
                                                        (frame.shape[0]*frame.shape[1]), 5)})
            anomaly_score, heat = (None, None)
            if anomaly is not None and quality['passed']:
                anomaly_score, heat = anomaly.score(frame)
            anomaly_alarm = anomaly is not None and anomaly_score is not None and anomaly_score > anomaly.threshold
            decision = choose_decision(quality, defects, anomaly_alarm, anomaly is not None)
            event_id = str(uuid.uuid4())
            event = {'event_id': event_id, 'timestamp_utc': datetime.now(timezone.utc).isoformat(),
                     'product_id': args.product, 'quality': quality, 'defects': defects,
                     'anomaly_score': anomaly_score, 'anomaly_alarm': bool(anomaly_alarm),
                     'decision': decision, 'model_sha': checksum}
            sop = make_advice(event)
            duration_ms = (time.perf_counter() - start)*1000.
            db.execute('INSERT INTO inspections VALUES (?,?,?,?,?,?,?,?,?,?,?,?)',
                       (event_id, event['timestamp_utc'], args.product, decision,
                        json.dumps(defects), anomaly_score, json.dumps(quality), checksum,
                        duration_ms, json.dumps(sop), None, None))
            db.commit()
            print(json.dumps({'id': event_id, 'decision': decision, 'defects': defects,
                              'anomaly': anomaly_score, 'latency_ms': round(duration_ms, 1),
                              'sop': sop}, default=float))
            if pool and decision not in ('ACCEPT_CANDIDATE',) and (pending is None or pending.done()):
                pending = pool.submit(advisory_job, event_id, make_prompt(event, sop))
            if args.save_images:
                cv2.imwrite(str(args.save_images / f'{event_id}.jpg'), frame)
            if args.view:
                display = result.plot() if result is not None else frame.copy()
                cv2.putText(display, decision, (8, 25), cv2.FONT_HERSHEY_SIMPLEX,
                            .8, (0, 0, 255), 2, cv2.LINE_AA)
                cv2.imshow('Offline Industry 4.0 QC (q=quit)', display)
                if cv2.waitKey(1) & 0xFF == ord('q'): break
            if single_image: break
    finally:
        if capture is not None: capture.release()
        if args.view: cv2.destroyAllWindows()
        if pool: pool.shutdown(wait=True)
        db.close()


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/industry4_qc/operator_feedback.py
#!/usr/bin/env python3
"""Human feedback on locally logged inspection events; does NOT update model automatically."""
import argparse
import csv
import sqlite3
from pathlib import Path

VALID = ['confirmed_defect', 'false_alarm', 'missed_defect', 'confirmed_good', 'uncertain']

def main():
    p=argparse.ArgumentParser()
    p.add_argument('--db',default='inspections.sqlite3')
    p.add_argument('--list',action='store_true',help='Show latest 15 inspections')
    p.add_argument('--event-id',help='Inspection UUID from event log')
    p.add_argument('--verdict',choices=VALID)
    p.add_argument('--export',help='Export human-labeled events as review CSV')
    args=p.parse_args()
    db=sqlite3.connect(args.db)
    if args.list:
        for row in db.execute('SELECT event_id,timestamp_utc,decision,defect_json,operator_feedback FROM inspections ORDER BY timestamp_utc DESC LIMIT 15'):
            print(*row,sep=' | ')
    if args.event_id or args.verdict:
        if not (args.event_id and args.verdict): p.error('Provide BOTH --event-id and --verdict')
        row=db.execute('SELECT event_id FROM inspections WHERE event_id=?',(args.event_id,)).fetchone()
        if row is None: p.error('Unknown event ID')
        db.execute('UPDATE inspections SET operator_feedback=? WHERE event_id=?',(args.verdict,args.event_id))
        db.commit(); print('Recorded feedback; no automatic retraining:',args.event_id,args.verdict)
    if args.export:
        rows=list(db.execute('SELECT event_id,timestamp_utc,product_id,decision,defect_json,anomaly_score,model_sha,operator_feedback FROM inspections WHERE operator_feedback IS NOT NULL'))
        with open(args.export,'w',newline='') as out:
            writer=csv.writer(out)
            writer.writerow(['event_id','timestamp_utc','product_id','decision','defect_json','anomaly_score','model_sha','operator_feedback'])
            writer.writerows(rows)
        print('Wrote human-review export:',args.export,'rows:',len(rows))
    db.close()
if __name__=='__main__': main()

## 13. Bundle download

This cell packages the detector ONNX, PyTorch checkpoint, optional anomaly ONNX/bank, Pi reference application, manifest, and sample event JSON. The package contains **no Llama model weights** because acceptance of Meta's license and ExecuTorch `.pte` export/build is a separate deployment process.

In [ ]:
import zipfile
artifact_zip=ROOT/'edge_deployment_bundle.zip'
files_to_package=['yolo_detector.onnx','yolo_detector_best.pt','manifest.json','edge_inspector.py',
                  'normal_bank.npz','mobilenet_patch_encoder.onnx','magnetic_tile_classifier.onnx','sample_llama_prompt.json', 'operator_feedback.py']
with zipfile.ZipFile(artifact_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for name in files_to_package:
        file=ROOT/name
        if file.exists(): z.write(file,arcname=name)
print('Prepared',artifact_zip, 'Size MB:',round(artifact_zip.stat().st_size/1e6,1))
try:
    from google.colab import files
    files.download(str(artifact_zip))
except ImportError:
    print('Outside Colab: retrieve the ZIP at',artifact_zip)